# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [5]:
%pip install --upgrade nbformat ipython

Defaulting to user installation because normal site-packages is not writeable
  Using cached nbformat-5.11.1-py3-none-any.whl.metadata (3.7 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.9.1-cp313-cp313-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.11.1-py3-none-any.whl (79 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-2026.9.1-cp313-cp313-win_amd64.whl (222 kB)

   ----- ---------------------------------- 1/7 [fastjsonschema]
   ----- ---------------------------------- 1/7 [fastjsonschema]
   -----


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: C:\Users\Chareesh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import nbformat
import IPython

print("nbformat version:", nbformat.__version__)
print("IPython version:", IPython.__version__)
print("Installation successful!")

nbformat version: 5.11.1
IPython version: 9.18.1
Installation successful!


In [3]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A line chart was my choice because it effectively illustrates the changes in population growth over time across the six continents. To make the continents comparable despite their different starting population sizes, I used a population index with 1970 as the baseline (100). I employed various color-blind-safe shades to represent the continents, added clear labels to the axes, and included an informative title.

In [4]:

# Q1: Which continent is growing fastest?

# Calculate total population for each continent and year
continent_pop = (
    df_long.groupby(['Continent', 'Year'], as_index=False)['Population']
    .sum()
)

# Get each continent's 1970 population as its baseline
baseline = (
    continent_pop[continent_pop['Year'] == 1970]
    [['Continent', 'Population']]
    .rename(columns={'Population': 'Population_1970'})
)

continent_pop = continent_pop.merge(baseline, on='Continent')

# Calculate population index (1970 = 100)
continent_pop['Population Index'] = (
    continent_pop['Population'] /
    continent_pop['Population_1970'] * 100
)

fig = px.line(
    continent_pop,
    x='Year',
    y='Population Index',
    color='Continent',
    markers=True,
    title='Africa Has Grown Fastest Relative to Its 1970 Population',
    labels={
        'Year': 'Year',
        'Population Index': 'Population Index (1970 = 100)',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    width=1000,
    height=600,
    yaxis_title='Population Index (1970 = 100)',
    xaxis_title='Year'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show(renderer="browser")


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A horizontal bar chart was my choice because it allows for easy comparison and ranking of the 15 countries that saw the largest population increases from 1970 to 2022. I employed various colour-blind-safe colors to represent continents, which helps in easily identifying African countries. The chart is organized in descending order, featuring countries with the largest population increases at the top.

In [5]:

# Q2: Which countries added the most people?

# Calculate population growth from 1970 to 2022
df['Population Added'] = (
    df['2022 Population'] - df['1970 Population']
)

# Select the 15 countries with the largest population increase
top15 = df.nlargest(15, 'Population Added').copy()

# Count how many of the top 15 countries are in Africa
africa_count = (top15['Continent'] == 'Africa').sum()

# Sort for a horizontal bar chart
top15 = top15.sort_values('Population Added', ascending=True)

# Create the chart
fig = px.bar(
    top15,
    x='Population Added',
    y='Country',
    color='Continent',
    orientation='h',
    title='India and China Lead Population Growth (1970–2022)',
    labels={
        'Population Added': 'Population Added (people)',
        'Country': 'Country',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

# Improve chart layout
fig.update_layout(
    title_x=0.5,
    width=1100,
    height=700,
    legend_title_text='Continent',
    xaxis_tickformat=',',
    yaxis_title='',
    xaxis_title='Population Added (people)'
)

# Improve hover information
fig.update_traces(
    hovertemplate=(
        '<b>%{y}</b><br>'
        'People added: %{x:,.0f}'
        '<extra></extra>'
    )
)

fig.show(renderer="browser")

# Display the result
print("Number of African countries in the top 15:", africa_count)

display(
    top15[
        ['Country', 'Continent', 'Population Added']
    ].sort_values(
        'Population Added', ascending=False
    ).reset_index(drop=True)
)


Number of African countries in the top 15: 4


,Country,Continent,Population Added
0,India,Asia,859671872
1,China,Asia,603352887
2,Pakistan,Asia,176533990
3,Nigeria,Africa,162971948
4,Indonesia,Asia,160272945
5,United States,North America,137961517
6,Brazil,South America,118943623
7,Bangladesh,Asia,103644512
8,Ethiopia,Africa,95071678
9,DR Congo,Africa,78858479


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A scatter plot was my choice since it illustrates how two numerical variables, land area and population, are related. The continent is represented by color, and selected countries are marked to showcase interesting examples.I applied logarithmic scales since the two variables exhibit significant disparities across countries. represented by color, and selected countries are marked to showcase interesting examples.

While there is a positive correlation between land area and population, it's not a strict rule. The correlation is around 0.45. Nations like Bangladesh have a huge population compared to their small land area, while Canada and Russia, despite their vast territories, have populations that are much smaller than those of India and China.

In [6]:

# Q3: Relationship between land area and population

# Select countries to label as useful reference points
label_countries = [
    'China', 'India', 'United States', 'Russia',
    'Canada', 'Bangladesh', 'Singapore', 'Australia'
]

df['Label'] = np.where(
    df['Country'].isin(label_countries),
    df['Country'],
    ''
)

fig = px.scatter(
    df,
    x='Area',
    y='2022 Population',
    color='Continent',
    hover_name='Country',
    text='Label',
    log_x=True,
    log_y=True,
    title='Land Area and Population Are Related, but Not Directly',
    labels={
        'Area': 'Land Area (km²)',
        '2022 Population': '2022 Population',
        'Continent': 'Continent'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig.update_traces(
    marker=dict(size=9, opacity=0.75),
    textposition='top center'
)

fig.update_layout(
    title_x=0.5,
    width=1050,
    height=700
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show(renderer="browser")


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A violin plot was my choice because it illustrates the distribution and spread of population density across each continent, rather than just presenting an average. The embedded box plot aids in comparing the central values ​​and the spread. A logarithmic scale was employed due to the significant variation in population density across countries.

Continents can have highly variable population densities. In this dataset, North America boasts the highest median density, with Asia and Europe following. The median density in South America is the lowest. Certain countries and territories have densities that are extremely high compared to the rest of their continent.

In [7]:

# Q4: Population density across continents

fig = px.violin(
    df,
    x='Continent',
    y='Density',
    color='Continent',
    box=True,
    points=False,
    title='Population Density Varies Widely Within Each Continent',
    labels={
        'Continent': 'Continent',
        'Density': 'Population Density (people per km²)'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    width=1050,
    height=650,
    showlegend=False
)

fig.update_yaxes(
    type='log',
    showgrid=False,
    title='Population Density (people per km², log scale)'
)

fig.update_xaxes(showgrid=False)

fig.show(renderer="browser")


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A choropleth map was my choice since the question is about the geographical areas of population growth and decline. Every country is shaded based on how much its population percentage changed from 2000 to 2022. Using a diverging color scale makes it easy to tell the difference between positive growth and population decline.

Many countries, especially in Africa and Asia, experienced population growth. Qatar saw the largest percentage increase in this dataset, at around 317%. Population declines were observed in several European countries and island territories. The Northern Mariana Islands saw the biggest drop, at around 38%.

In [8]:

# Q5: Percentage population change between 2000 and 2022

df['Population Change (%)'] = (
    (df['2022 Population'] - df['2000 Population'])
    / df['2000 Population'] * 100
)

fig = px.choropleth(
    df,
    locations='CCA3',
    locationmode='ISO-3',
    color='Population Change (%)',
    hover_name='Country',
    color_continuous_scale='PuOr',
    color_continuous_midpoint=0,
    title='Population Growth and Decline Across Countries (2000–2022)',
    labels={
        'Population Change (%)': 'Population Change (%)'
    },
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=650,
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    )
)

fig.show(renderer="browser")

# Display countries with the largest increases and decreases
print("Top 10 countries by percentage growth:")
display(
    df.nlargest(10, 'Population Change (%)')[
        ['Country', 'Continent', 'Population Change (%)']
    ].reset_index(drop=True)
)

print("Top 10 countries with the largest percentage decline:")
display(
    df.nsmallest(10, 'Population Change (%)')[
        ['Country', 'Continent', 'Population Change (%)']
    ].reset_index(drop=True)
)


Top 10 countries by percentage growth:


,Country,Continent,Population Change (%)
0,Qatar,Asia,317.242239
1,United Arab Emirates,Asia,188.249439
2,Equatorial Guinea,Africa,144.520327
3,Turks and Caicos Islands,North America,143.827358
4,Niger,Africa,125.490255
5,Jordan,Asia,123.209664
6,Kuwait,Asia,120.624880
7,Angola,Africa,117.084619
8,Chad,Africa,114.590398
9,Western Sahara,Africa,113.032270


Top 10 countries with the largest percentage decline:


,Country,Continent,Population Change (%)
0,Northern Mariana Islands,Oceania,-38.321840
1,American Samoa,Oceania,-23.968745
2,Lithuania,Europe,-23.601880
3,Marshall Islands,Oceania,-23.338374
4,Moldova,Europe,-23.016822
5,Latvia,Europe,-22.648786
6,Bosnia and Herzegovina,Europe,-22.630888
7,Vatican City,Europe,-21.658986
8,Wallis and Futuna,Oceania,-21.401888
9,Ukraine,Europe,-18.776723


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

opted for a treemap since it can show hierarchical data all in one view. them. 2022, allowing for easy comparisons between continents and countries.

With around 4.72 billion people in 2022, Asia is home to the largest population. trillion. about 743 million. up a significant part of the Asian population.

In [9]:

# Q6: Population distribution across continents and countries

fig = px.treemap(
    df,
    path=['Continent', 'Country'],
    values='2022 Population',
    color='Continent',
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='Asia Contains the Largest Share of the World Population',
    labels={
        '2022 Population': '2022 Population',
        'Continent': 'Continent',
        'Country': 'Country'
    },
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=700
)

fig.update_traces(
    textinfo='label+percent parent',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Population: %{value:,.0f}<br>'
        'Share of parent: %{percentParent:.1%}'
        '<extra></extra>'
    )
)

fig.show(renderer="browser")


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

A 100% stacked bar chart is my choice because it shows how the share of the world's population is divided among Africa, Asia, Europe, andthe rest of the world over time. Since each bar represents the entire world population (100%), it is easy to compare changes in regional shares.

Africa's proportion of the global population grew from about 9.9% in 1970 to 17.9% by 2022. Asia continued to be the largest region, with its share rising slightly from around 58.1% to 59.2%. Europe's share fell from about 17.8% to 9.3%. This indicates that Africa's share is increasing as Europe's share declines.

In [10]:

# Q7: Share of world population over time

# Group the continents requested in the question
df_long['Group'] = np.where(
    df_long['Continent'].isin(['Africa', 'Asia', 'Europe']),
    df_long['Continent'],
    'Rest of the world'
)

# Calculate population totals for each group and year
grouped = (
    df_long.groupby(['Year', 'Group'], as_index=False)['Population']
    .sum()
)

# Calculate each group's percentage of world population
grouped['World Share (%)'] = (
    grouped['Population']
    / grouped.groupby('Year')['Population'].transform('sum')
    * 100
)

# Create a 100% stacked bar chart
fig = px.bar(
    grouped,
    x='Year',
    y='World Share (%)',
    color='Group',
    barmode='stack',
    title='Africa’s Share of World Population Is Rising',
    labels={
        'Year': 'Year',
        'World Share (%)': 'Share of World Population (%)',
        'Group': 'Region'
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    template='plotly_white'
)

fig.update_layout(
    title_x=0.5,
    width=1100,
    height=650,
    yaxis=dict(range=[0, 100]),
    xaxis_title='Year',
    yaxis_title='Share of World Population (%)'
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show(renderer="browser")
